In [2]:
# ==============================================================================
# STEP 0: DATASET SOURCE CONFIGURATION & PATH VERIFICATION
# ==============================================================================

DATASET_META = {
    "dataset_name": "Nail Disease Image Classification Dataset",
    "dataset_url": "https://www.kaggle.com/datasets/josephrasanjana/nail-disease-image-classification-dataset",
    "kaggle_slug": "josephrasanjana/nail-disease-image-classification-dataset",
    "kaggle_input_path": "/kaggle/input/datasets/josephrasanjana/nail-disease-image-classification-dataset",
    "surface": "Nail",
    "target_classes": ["Healthy", "Psoriasis", "Onychomycosis"]
}

from pathlib import Path

DATASET_ROOT = Path(DATASET_META["kaggle_input_path"])

# Secondary check in case Kaggle mounted without the /datasets/ prefix
if not DATASET_ROOT.exists():
    DATASET_ROOT = Path(f"/kaggle/input/{DATASET_META['kaggle_slug'].split('/')[-1]}")

assert DATASET_ROOT.exists(), (
    f" Dataset not found at: {DATASET_META['kaggle_input_path']}\n"
    f"Please attach the dataset from Kaggle: {DATASET_META['dataset_url']}"
)

print(f" Verified: {DATASET_META['dataset_name']}")
print(f" Source URL: {DATASET_META['dataset_url']}")
print(f" Mounted Path: {DATASET_ROOT}")

 Verified: Nail Disease Image Classification Dataset
 Source URL: https://www.kaggle.com/datasets/josephrasanjana/nail-disease-image-classification-dataset
 Mounted Path: /kaggle/input/datasets/josephrasanjana/nail-disease-image-classification-dataset


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
#Step 1 — Locate Dataset in Kaggle Environment
from pathlib import Path

# Check what input directories exist
for p in Path('/kaggle/input/datasets/josephrasanjana/nail-disease-image-classification-dataset').iterdir():
    print(p)

In [ ]:
#Step 2 — Explore Folder Structure & Class Counts
from pathlib import Path
from collections import Counter, defaultdict

# Adjust folder name if Kaggle mounted it under a slightly different slug
DATASET_ROOT = Path('/kaggle/input/nail-disease-image-classification-dataset')

# If nested under another root folder, locate it dynamically
if not DATASET_ROOT.exists():
    for p in Path('/kaggle/input').rglob('*'):
        if 'nail' in p.name.lower() and p.is_dir():
            DATASET_ROOT = p
            break

print("Using Dataset Root:", DATASET_ROOT)

# Check splits / subfolders
counts = defaultdict(Counter)
for item in DATASET_ROOT.rglob('*'):
    if item.is_file() and item.suffix.lower() in ['.jpg', '.jpeg', '.png']:
        split_name = item.parent.parent.name if item.parent.parent != DATASET_ROOT else "root"
        class_name = item.parent.name
        counts[split_name][class_name] += 1

for split, cls_dict in counts.items():
    print(f"\n--- Split / Folder: {split} ---")
    for cls, n in sorted(cls_dict.items(), key=lambda x: -x[1]):
        print(f"  {cls}: {n}")

In [ ]:
#Step 3 — Filename Inspection & Deduplication / Hash Leakage Check
import hashlib
from pathlib import Path
from collections import defaultdict

DATASET_ROOT = Path('/kaggle/input/datasets/josephrasanjana/nail-disease-image-classification-dataset/nail_disease_dataset')
if not DATASET_ROOT.exists():
    DATASET_ROOT = Path('/kaggle/input/datasets/josephrasanjana/nail-disease-image-classification-dataset')

def compute_md5(file_path: Path) -> str:
    hasher = hashlib.md5()
    with open(file_path, 'rb') as f:
        hasher.update(f.read())
    return hasher.hexdigest()

# Sample 5 filenames per class to inspect naming convention
print("--- Sample Filenames ---")
for split in ['train', 'test']:
    split_dir = DATASET_ROOT / split
    if split_dir.is_dir():
        for cls_dir in split_dir.iterdir():
            if cls_dir.is_dir():
                samples = [f.name for f in list(cls_dir.glob('*'))[:3]]
                print(f"{split}/{cls_dir.name}: {samples}")

# Hash collisions and cross-split check
train_hashes = defaultdict(set)
test_hashes = defaultdict(set)
all_hashes = defaultdict(list)

for split in ['train', 'test']:
    split_dir = DATASET_ROOT / split
    if not split_dir.is_dir():
        continue
    for cls_dir in split_dir.iterdir():
        if not cls_dir.is_dir():
            continue
        for f in cls_dir.glob('*'):
            if f.is_file() and f.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                h = compute_md5(f)
                all_hashes[h].append((split, cls_dir.name, f.name))
                if split == 'train':
                    train_hashes[cls_dir.name].add(h)
                else:
                    test_hashes[cls_dir.name].add(h)

# Duplicate check
exact_dupes = {k: v for k, v in all_hashes.items() if len(v) > 1}
print(f"\nExact duplicate image files found across dataset: {len(exact_dupes)}")

# Cross-split leakage check
for cls in ['onychomycosis', 'psoriasis', 'healthy']:
    overlap = train_hashes[cls] & test_hashes[cls]
    print(f"{cls}: {len(overlap)} exact images appear in BOTH train and test")

In [ ]:
#Step 4 — Build the Image-Level Table

import pandas as pd
from PIL import Image

records = []

for split in ['train', 'test']:
    split_dir = DATASET_ROOT / split
    if not split_dir.is_dir():
        continue
    for cls_dir in split_dir.iterdir():
        if not cls_dir.is_dir():
            continue
        
        category = cls_dir.name.capitalize()
        
        for f in cls_dir.glob('*'):
            if not f.is_file() or f.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
                continue
            
            img_hash = compute_md5(f)
            
            try:
                with Image.open(f) as im:
                    width, height = im.size
            except Exception as e:
                width, height = None, None
                print(f"⚠️ Error reading {f.name}: {e}")
                
            records.append({
                'source_dataset': 'josephrasanjana-nail-disease',
                'image_id': f.stem,
                'file_name': f.name,
                'file_path': str(f),
                'width': width,
                'height': height,
                'Category': category,
                'original_split': split,
                'img_hash': img_hash
            })

df_nail3 = pd.DataFrame(records)
print(f"\nTotal records collected: {len(df_nail3)}")
print(df_nail3['Category'].value_counts())

In [ ]:
#Step 5 — Recover Base Names & Deduplicate by Source Group

import re
import pandas as pd

def extract_base_source(filename: str) -> str:
    """
    Strips Roboflow augmentation hashes (_jpg.rf.<hash> or _png_jpg.rf.<hash>)
    while retaining intact camera filenames.
    """
    # Remove Roboflow hash pattern
    cleaned = re.sub(r'(_(?:jpg|jpeg|png))?\.rf\.[a-f0-9]+\.(jpg|jpeg|png)$', '', filename, flags=re.IGNORECASE)
    # Remove standard extension for grouping
    cleaned = re.sub(r'\.(jpg|jpeg|png)$', '', cleaned, flags=re.IGNORECASE)
    return cleaned

df_nail3['base_source_id'] = df_nail3['file_name'].apply(extract_base_source)
df_nail3['source_group_id'] = df_nail3['Category'] + "__" + df_nail3['base_source_id']

# Exact MD5 deduplication first
df_dedup_md5 = df_nail3.drop_duplicates(subset='img_hash', keep='first').copy()

# Deduplicate by base source group (one representative image per unique source)
df_nail3_clean = df_dedup_md5.drop_duplicates(subset='source_group_id', keep='first').reset_index(drop=True)

print(f"Total raw files: {len(df_nail3)}")
print(f"After MD5 dedup: {len(df_dedup_md5)}")
print(f"After source group dedup (true unique images): {len(df_nail3_clean)}")
print("\n--- Cleaned Per-Class Distribution ---")
print(df_nail3_clean['Category'].value_counts())

In [ ]:
#Step 6 — Cross-Class Collision & Sanity Checks
from collections import defaultdict

base_to_classes = defaultdict(set)
for _, row in df_nail3_clean.iterrows():
    base_to_classes[row['base_source_id']].add(row['Category'])

cross_class = {k: v for k, v in base_to_classes.items() if len(v) > 1}
print(f"Source images mapped to multiple classes: {len(cross_class)}")
if cross_class:
    print("Collisions:", cross_class)

print("\nUnreadable/Corrupt images:", df_nail3_clean['width'].isna().sum())
print("\nResolution summary:")
print(df_nail3_clean[['width', 'height']].describe())

In [ ]:
#Step 7 — Standardized Schema Mapping & CSV Export
nail3_standardized_df = df_nail3_clean.assign(
    **{'Clinical Diagnosis': df_nail3_clean['Category']}
)[['source_dataset', 'image_id', 'file_name', 'width', 'height', 'Category', 'Clinical Diagnosis']]

output_path = '/kaggle/working/nail_psoriasis_standardized_df.csv'
nail3_standardized_df.to_csv(output_path, index=False)

print(f"\nSaved {len(nail3_standardized_df)} clean rows to {output_path}.")
nail3_standardized_df.head()

In [ ]:
#Step 8 — Drop Colliding Ambiguous Base Names & Finalize Export
import pandas as pd

# Load saved intermediate df
df = pd.read_csv('/kaggle/working/nail_psoriasis_standardized_df.csv')

# Identify colliding base_source_ids
colliding_bases = {'2', '3'}

# Filter out files whose base name was exactly '2' or '3'
clean_mask = ~df['file_name'].str.match(r'^(2|3)\.(jpg|jpeg|png)$', case=False)
nail3_final_df = df[clean_mask].reset_index(drop=True)

print(f"Dropped {len(df) - len(nail3_final_df)} ambiguous collision rows.")
print(f"Final clean dataset count: {len(nail3_final_df)}")
print("\nFinal clean per-class counts:")
print(nail3_final_df['Category'].value_counts())

# Save final standardized dataframe
output_path = '/kaggle/working/nail_psoriasis_standardized_df.csv'
nail3_final_df.to_csv(output_path, index=False)
print(f"Updated {output_path}")